# Module 1 - SQL Warehouse, pricing and data profiling

![SQL Warehouse cost decision](../assets/images/sql_warehouse_cost_decision.png)

This module starts the RetailHub case. The goal is not only to run SQL, but to
understand which warehouse and Power BI mode will be cost-safe for the report.

In [ ]:
%run ../setup/00_setup

## Real Databricks UI context

Use the source screenshots below as short visual anchors before the hands-on
part. They come from the base `Databricks-Data-Analyst` training.

![Databricks SQL Editor](../assets/images/source_sql_editor.png)

![Catalog Explorer hierarchy](../assets/images/source_catalog_explorer_hierarchy.png)

## RetailHub source map

![RetailHub source map](../assets/images/retailhub_source_map.png)

## Business question

RetailHub wants an executive KPI dashboard. Before building Gold tables, we
need to answer:

- What data exists?
- What is the grain?
- Which columns drive filters and joins?
- What can make the report expensive?

## Runtime pre-check

If this fails, run:

1. `setup/00_pre_config.ipynb`
2. `data/generate_training_dataset.ipynb`

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
    f"{GOLD}.fact_sales",
]

missing = [table for table in required_tables if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing tables. Run setup and dataset generator first: " + ", ".join(missing))

print("[OK] Required tables exist")

In [ ]:
spark.sql(f"SHOW TABLES IN {SILVER}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {GOLD}").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT 'customers' AS table_name, COUNT(*) AS rows FROM {SILVER}.customers
UNION ALL SELECT 'products', COUNT(*) FROM {SILVER}.products
UNION ALL SELECT 'sales_orders', COUNT(*) FROM {SILVER}.sales_orders
UNION ALL SELECT 'order_lines', COUNT(*) FROM {SILVER}.order_lines
""").show()

## Pricing and cost discussion

Use the official Databricks pricing page or calculator during delivery. Do not
treat values copied into training materials as permanent.

Trainer asset to refresh before delivery:

`assets/images/databricks_pricing_YYYY-MM-DD.png`

Decision questions:

- Is Import mode enough for the report?
- Which pages need live data?
- Does DirectQuery query Silver or Gold?
- What auto-stop is acceptable for a live demo?

## Warehouse sizing discussion

Use this as a guided conversation, not as a static pricing lecture.

| Scenario | Candidate mode | Cost risk | Mitigation |
|---|---|---|---|
| Daily executive dashboard | Import | refresh window too wide | incremental refresh, monthly aggregate |
| Operational live page | DirectQuery | every slicer interaction sends SQL | Gold aggregate, fewer visuals, query profile |
| Ad-hoc analyst exploration | SQL Editor / Notebook | large scans | filters, sample first, select needed columns |
| Training demo | Serverless SQL Warehouse | idle time | short auto-stop, prepared objects |

Before delivery, refresh the current official pricing screenshot and place it
as `assets/images/databricks_pricing_YYYY-MM-DD.png`.

## Data profiling

In [ ]:
spark.sql(f"""
SELECT
  status,
  COUNT(DISTINCT order_id) AS orders
FROM {SILVER}.sales_orders
GROUP BY status
ORDER BY orders DESC
""").show()

In [ ]:
spark.sql(f"""
SELECT
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date,
  COUNT(*) AS rows,
  COUNT(DISTINCT order_id) AS orders,
  COUNT(DISTINCT product_id) AS products
FROM {SILVER}.order_lines
""").show()

In [ ]:
spark.sql(f"""
SELECT
  channel,
  region,
  COUNT(DISTINCT order_id) AS orders,
  ROUND(SUM(line_revenue), 2) AS line_revenue
FROM {SILVER}.order_lines
GROUP BY channel, region
ORDER BY line_revenue DESC
""").show(truncate=False)

In [ ]:
spark.sql(f"""
SELECT
  category,
  COUNT(*) AS lines,
  COUNT(DISTINCT product_id) AS products,
  ROUND(AVG(unit_price), 2) AS avg_price,
  ROUND(AVG(unit_cost), 2) AS avg_cost
FROM {SILVER}.order_lines
GROUP BY category
ORDER BY lines DESC
""").show(truncate=False)

## First risk scan

This is the moment to show why Gold exists. The source is usable, but not yet a
trusted BI contract.

In [ ]:
spark.sql(f"""
SELECT 'invalid_status' AS issue, COUNT(*) AS rows FROM {SILVER}.sales_orders
WHERE status NOT IN ('Completed','Cancelled','Returned')
UNION ALL
SELECT 'future_orders', COUNT(*) FROM {SILVER}.sales_orders
WHERE order_date > current_date()
UNION ALL
SELECT 'missing_prices', COUNT(*) FROM {SILVER}.order_lines
WHERE unit_price IS NULL
UNION ALL
SELECT 'duplicate_line_ids', COUNT(*) - COUNT(DISTINCT line_id) FROM {SILVER}.order_lines
""").show(truncate=False)

## Artifact: first data map

Fill this in during the session:

| Object | Grain | Business use | Risk |
|---|---|---|---|
| `silver.sales_orders` | one row per order | status, date, customer | status definitions |
| `silver.order_lines` | one row per order line | revenue, margin, quantity | missing prices, duplicates |
| `gold.fact_sales` | one row per order line | BI fact | must be validated |

## Mini exercise: warehouse decision

In pairs, decide the baseline for the executive dashboard:

| Decision | Your answer |
|---|---|
| Import or DirectQuery by default? | |
| Which Gold object should feed summary pages? | |
| What warehouse auto-stop would you use for a live demo? | |
| Which query should never be executed by a Power BI visual? | |

Trainer note: this section can easily take 15-20 minutes when participants
compare cost, freshness and usability trade-offs.